# 01 · Data Collection

Collect raw match data from two sources and save it **untouched** to `data/raw/`.

| Source | What we take from it | How we get it |
|---|---|---|
| **DataHub** (mirror of football-data.co.uk) | Results, shots, fouls, cards | Downloaded automatically |
| **FBref** | Attendance, venue, referee | Saved manually from the browser (FBref blocks scripts) |

**Scope:** top 5 European leagues, 2017–18 to 2025–26.

**Test season: Premier League 2019–20.** Fans for most of the season, then empty stadiums after the June 2020 restart, so attendance goes from full to blank inside one file.

Cleaning and merging happen in `02_clean`. This notebook only collects and takes a first look.

In [1]:
import time
from io import StringIO
from pathlib import Path

import pandas as pd
import requests

RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)

SEASONS = range(2017, 2026)   # 2017-18 ... 2025-26
DELAY = 1                     # seconds between DataHub downloads

## League codes

Each source uses its own codes. `datahub` = DataHub dataset name, `fbref_id` / `fbref_name` = FBref URL parts.

In [2]:
LEAGUES = {
    "EPL":        {"datahub": "english-premier-league",  "fbref_id": 9,  "fbref_name": "Premier-League"},
    "La_Liga":    {"datahub": "spanish-la-liga",        "fbref_id": 12, "fbref_name": "La-Liga"},
    "Serie_A":    {"datahub": "italian-serie-a",        "fbref_id": 11, "fbref_name": "Serie-A"},
    "Bundesliga": {"datahub": "german-bundesliga",      "fbref_id": 20, "fbref_name": "Bundesliga"},
    "Ligue_1":    {"datahub": "french-ligue-1",         "fbref_id": 13, "fbref_name": "Ligue-1"},
}

def season_codes(start_year):
    """2019 -> ('2019-2020', '1920')  (FBref format, football-data format)"""
    end = start_year + 1
    return f"{start_year}-{end}", f"{str(start_year)[2:]}{str(end)[2:]}"

## Helper: download once, then read from disk

Every DataHub file is cached in `data/raw/`. Rerunning the notebook won't download again unless you delete the file or pass `force=True`.

In [3]:
def fetch(url, path, force=False):
    """Download url to path once; afterwards read it from disk."""
    path = Path(path)
    if path.exists() and not force:
        print(f"cached   {path.name}")
        return path.read_bytes()
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    path.write_bytes(r.content)
    print(f"download {path.name}  ({len(r.content)/1024:.0f} KB)")
    time.sleep(DELAY)
    return r.content

## Source 1: football-data.co.uk (via DataHub)

football-data.co.uk is blocked on our network, so we use [DataHub](https://datahub.io/football), which hosts the same football-data.co.uk files, one CSV per league per season.

Column key: `HF/AF` fouls, `HY/AY` yellows, `HR/AR` reds, `HS/AS` shots, `HST/AST` shots on target.

In [4]:
def get_football_data(league, start_year, force=False):
    _, fd_season = season_codes(start_year)          # 2019 -> '1920'
    slug = LEAGUES[league]["datahub"]
    url = f"https://datahub.io/football/{slug}/_r/-/season-{fd_season}.csv"
    path = RAW / f"fd_{league}_{start_year}.csv"
    raw = fetch(url, path, force)
    df = pd.read_csv(StringIO(raw.decode("utf-8", errors="replace")))
    df["league"] = league
    df["season"] = start_year
    return df

fd = get_football_data("EPL", 2019)
print(fd.shape)   # expect 380 rows
fd.head()

cached   fd_EPL_2019.csv
(380, 24)


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,...,HF,AF,HC,AC,HY,AY,HR,AR,league,season
0,2019-08-09,Liverpool,Norwich,4,1,H,4,0,H,M Oliver,...,9,9,11,2,0,2,0,0,EPL,2019
1,2019-08-10,West Ham,Man City,0,5,A,0,1,A,M Dean,...,6,13,1,1,2,2,0,0,EPL,2019
2,2019-08-10,Bournemouth,Sheffield United,1,1,D,0,0,D,K Friend,...,10,19,3,4,2,1,0,0,EPL,2019
3,2019-08-10,Burnley,Southampton,3,0,H,0,0,D,G Scott,...,6,12,2,7,0,0,0,0,EPL,2019
4,2019-08-10,Crystal Palace,Everton,0,0,D,0,0,D,J Moss,...,16,14,6,2,2,1,0,1,EPL,2019


In [5]:
# the columns we actually care about
keep = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Referee",
        "HS", "AS", "HST", "AST", "HF", "AF", "HY", "AY", "HR", "AR"]
missing = [c for c in keep if c not in fd.columns]
print("missing columns:", missing or "none")
fd[keep].head()

missing columns: none


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,Referee,HS,AS,HST,AST,HF,AF,HY,AY,HR,AR
0,2019-08-09,Liverpool,Norwich,4,1,H,M Oliver,15,12,7,5,9,9,0,2,0,0
1,2019-08-10,West Ham,Man City,0,5,A,M Dean,5,14,3,9,6,13,2,2,0,0
2,2019-08-10,Bournemouth,Sheffield United,1,1,D,K Friend,13,8,3,3,10,19,2,1,0,0
3,2019-08-10,Burnley,Southampton,3,0,H,G Scott,10,11,4,3,6,12,0,0,0,0
4,2019-08-10,Crystal Palace,Everton,0,0,D,J Moss,6,10,2,3,16,14,2,1,0,1


## Source 2: FBref (attendance)

FBref's "Scores & Fixtures" page has one table with every match, including **Attendance**.

FBref's bot protection blocks Python (403 error), so the pages are **saved manually**:

1. Run the next cell to get the list of 45 URLs and file names.
2. Open each URL in Chrome, press **Ctrl+S**, choose **"Webpage, HTML Only"**, and save it in `data/raw/` with the file name shown.
3. The loader below reads the saved files. It never contacts FBref.

In [6]:
def fbref_url(league, start_year):
    fb_season, _ = season_codes(start_year)
    info = LEAGUES[league]
    return (f"https://fbref.com/en/comps/{info['fbref_id']}/{fb_season}/schedule/"
            f"{fb_season}-{info['fbref_name']}-Scores-and-Fixtures")

# which pages still need saving?
todo = [(lg, yr) for lg in LEAGUES for yr in SEASONS
        if not (RAW / f"fbref_{lg}_{yr}.html").exists()]
print(f"{45 - len(todo)}/45 saved\n")
for lg, yr in todo:
    print(f"fbref_{lg}_{yr}.html\n  {fbref_url(lg, yr)}")

45/45 saved



In [7]:
def get_fbref(league, start_year):
    path = RAW / f"fbref_{league}_{start_year}.html"
    if not path.exists():
        raise FileNotFoundError(f"Save this page as {path.name}:\n  {fbref_url(league, start_year)}")
    html = path.read_text(encoding="utf-8")
    # pick the fixtures table: the one with Home, Away and Attendance columns
    for t in pd.read_html(StringIO(html)):
        if {"Home", "Away", "Attendance"} <= set(t.columns):
            t["league"] = league
            t["season"] = start_year
            return t
    raise ValueError(f"No fixtures table in {path.name}. Was the page fully loaded when saved?")

fb = get_fbref("EPL", 2019)
print(fb.shape)
fb.head()

(419, 14)


,Wk,Day,Date,Time,Home,Score,Away,Attendance,Venue,Referee,Match Report,Notes,league,season
0,1.0,Fri,2019-08-09,20:00,Liverpool,4–1,Norwich City,53333.0,Anfield,Michael Oliver,Match Report,NaN,EPL,2019
1,1.0,Sat,2019-08-10,12:30,West Ham,0–5,Manchester City,59870.0,London Stadium,Mike Dean,Match Report,NaN,EPL,2019
2,1.0,Sat,2019-08-10,15:00,Burnley,3–0,Southampton,19784.0,Turf Moor,Graham Scott,Match Report,NaN,EPL,2019
3,1.0,Sat,2019-08-10,15:00,Watford,0–3,Brighton,20245.0,Vicarage Road Stadium,Craig Pawson,Match Report,NaN,EPL,2019
4,1.0,Sat,2019-08-10,15:00,Crystal Palace,0–0,Everton,25151.0,Selhurst Park,Jonathan Moss,Match Report,NaN,EPL,2019


## First look

Quick checks only, no fixing yet. Write down what you notice; each issue becomes a cleaning step in `02_clean`.

In [8]:
print("FBref columns:", fb.columns.tolist())
print()
print(fb.dtypes)

FBref columns: ['Wk', 'Day', 'Date', 'Time', 'Home', 'Score', 'Away', 'Attendance', 'Venue', 'Referee', 'Match Report', 'Notes', 'league', 'season']

Wk              float64
Day                 str
Date                str
Time                str
Home                str
Score               str
Away                str
Attendance      float64
Venue               str
Referee             str
Match Report        str
Notes           float64
league              str
season            int64
dtype: object


In [9]:
# FBref adds blank separator rows between matchweeks
fb_matches = fb.dropna(subset=["Home", "Away"])
print(f"{len(fb)} rows -> {len(fb_matches)} real matches   (football-data: {len(fd.dropna(subset=['HomeTeam']))})")

419 rows -> 380 real matches   (football-data: 380)


In [10]:
# the key moment: attendance before vs after the COVID restart
fb_matches = fb_matches.assign(Date=pd.to_datetime(fb_matches["Date"], errors="coerce"))
fb_matches["has_attendance"] = fb_matches["Attendance"].notna()
fb_matches.groupby(fb_matches["Date"].dt.to_period("M"))["has_attendance"].agg(["sum", "count"])

,sum,count
Date,,
2019-08,38,38
2019-09,32,32
2019-10,30,30
2019-11,36,36
2019-12,63,63
2020-01,41,41
2020-02,36,36
2020-03,12,12
2020-06,0,26


In [11]:
# team names: do the two sources agree?
fd_teams = set(fd["HomeTeam"].dropna())
fb_teams = set(fb_matches["Home"])
print("only in football-data:", sorted(fd_teams - fb_teams))
print("only in FBref:        ", sorted(fb_teams - fd_teams))

only in football-data: ['Leicester', 'Man City', 'Man United', 'Norwich']
only in FBref:         ['Leicester City', 'Manchester City', 'Manchester Utd', 'Norwich City']


## Things to note for cleaning

Findings from EPL 2019–20:

- Both sources have **380 matches**. FBref adds 39 blank separator rows between matchweeks, which need dropping.
- Attendance is already numeric (`float64`). Ghost games are **missing (NaN), not 0**: every match until March 2020 has attendance, and 0 of 92 matches in June–July 2020 do. Decide: fill with 0, or keep as missing plus a `ghost_game` flag?
- Dates use the same format (`yyyy-mm-dd`) in both sources.
- Team names differ, e.g. `Man United` vs `Manchester Utd`, `Leicester` vs `Leicester City`. More will appear in other leagues, so we need a name mapping.
- FBref referee names are full (`Michael Oliver`); DataHub's are short (`M Oliver`). Keep DataHub's, or use them to double-check the merge.
- Anything else surprising:

## Collect everything

Downloads the 45 DataHub files (about a minute) and reads the saved FBref pages. Any FBref page not saved yet is listed with its URL, so you can save it and rerun. Cached files are skipped.

In [12]:
RUN_ALL = True  # flip to True when ready

if RUN_ALL:
    counts, missing = [], []
    for league in LEAGUES:
        for year in SEASONS:
            row = {"league": league, "season": year}
            try:
                row["datahub"] = len(get_football_data(league, year))
            except Exception as e:
                print(f"FAILED DataHub {league} {year}: {e}")
            try:
                row["fbref"] = len(get_fbref(league, year).dropna(subset=["Home", "Away"]))
            except FileNotFoundError as e:
                missing.append(str(e))
            counts.append(row)

    print("\nFBref pages still to save:", len(missing))
    for m in missing:
        print(" ", m)

    # match counts should agree between the two sources
    summary = pd.DataFrame(counts).set_index(["league", "season"])
    summary["match"] = summary.get("datahub") == summary.get("fbref")
    display(summary)

cached   fd_EPL_2017.csv
cached   fd_EPL_2018.csv
cached   fd_EPL_2019.csv
cached   fd_EPL_2020.csv
cached   fd_EPL_2021.csv
cached   fd_EPL_2022.csv
cached   fd_EPL_2023.csv
cached   fd_EPL_2024.csv
cached   fd_EPL_2025.csv
cached   fd_La_Liga_2017.csv
cached   fd_La_Liga_2018.csv
cached   fd_La_Liga_2019.csv
cached   fd_La_Liga_2020.csv
cached   fd_La_Liga_2021.csv
cached   fd_La_Liga_2022.csv
cached   fd_La_Liga_2023.csv
cached   fd_La_Liga_2024.csv
cached   fd_La_Liga_2025.csv
cached   fd_Serie_A_2017.csv
cached   fd_Serie_A_2018.csv
cached   fd_Serie_A_2019.csv
cached   fd_Serie_A_2020.csv
cached   fd_Serie_A_2021.csv
cached   fd_Serie_A_2022.csv
cached   fd_Serie_A_2023.csv
cached   fd_Serie_A_2024.csv
cached   fd_Serie_A_2025.csv
cached   fd_Bundesliga_2017.csv
cached   fd_Bundesliga_2018.csv
cached   fd_Bundesliga_2019.csv
cached   fd_Bundesliga_2020.csv
cached   fd_Bundesliga_2021.csv
cached   fd_Bundesliga_2022.csv
cached   fd_Bundesliga_2023.csv
cached   fd_Bundesliga_2024.c

datahub  fbref  match
league     season                       
EPL        2017        380    380   True
           2018        380    380   True
           2019        380    380   True
           2020        380    380   True
           2021        380    380   True
           2022        380    380   True
           2023        380    380   True
           2024        380    380   True
           2025        380    380   True
La_Liga    2017        380    380   True
           2018        380    380   True
           2019        380    380   True
           2020        380    380   True
           2021        380    380   True
           2022        380    380   True
           2023        380    380   True
           2024        380    380   True
           2025        380    380   True
Serie_A    2017        380    380   True
           2018        380    380   True
           2019        380    380   True
           2020        380    380   True
           2021        380    380   True
           2022        380    381  False
           2023        380    380   True
           2024        380    380   True
           2025        380    380   True
Bundesliga 2017        306    308  False
           2018        306    308  False
           2019        306    308  False
           2020        306    308  False
           2021        306    308  False
           2022        306    308  False
           2023        306    308  False
           2024        306    308  False
           2025        306    308  False
Ligue_1    2017        380    382  False
           2018        380    382  False
           2019        279    380  False
           2020        380    382  False
           2021        380    382  False
           2022        380    380   True
           2023        306    308  False
           2024        306    308  False
           2025        306    308  False